# Swap control for the four open-weight agents (Modal GPUs, run from Colab)

Apart AI Collusion Sprint, Track 2. Shoumya Chowdhury and Anmita Das.

**What this does.** The report has a swap control (each firm reads the messages of a *different* run instead of its real rival's) for the two Claude models only. This notebook runs the same control for Qwen2.5-7B, Qwen2.5-32B, Phi-4 and Mistral-Nemo-12B with the same prompts, seeds and settings as the main runs (prompt 1, temperature 0.7, 200 reply tokens). It plays 40 to 50 runs for each of `honest_swap` and `collude_swap` per model, then compares them with the saved `honest` and `collude` runs. No Anthropic key is needed.

**Before you start (Colab left sidebar, key icon, "Secrets", turn Notebook access ON for each):**
- `MODAL_TOKEN_ID` and `MODAL_TOKEN_SECRET`: from your Modal profile (Settings, API Tokens), workspace `shoumyac`.
- `MODAL_SERVE_KEY`: the one line inside `work/.modal_key` on your computer. It must equal the value stored in your Modal secret `modal-serve`.
- Your Modal workspace must already have the secrets `huggingface` and `modal-serve` (it does, from the earlier runs).

**Rough cost.** About 5 to 15 minutes of GPU per model: 7B on A10G, Nemo and Phi-4 on L40S, 32B on A100 80GB. Expect a few US dollars in total. Each model is a separate cell, and a rerun resumes where it stopped.

**Files.** Upload `colab_bundle.zip` (same folder as this notebook) when asked. At the end the notebook downloads `swap_results.zip`; put it back in the `work` folder or send it to Claude.

In [ ]:
# 1. Install (numpy is pinned because the starter code was built against numpy 1.x)
!pip -q install modal openai "numpy==1.26.4" pandas pyyaml pydantic python-dotenv tqdm anthropic

In [ ]:
# 2. Secrets and token. Nothing is printed.
import os, pathlib
from google.colab import userdata
os.environ["MODAL_TOKEN_ID"] = userdata.get("MODAL_TOKEN_ID")
os.environ["MODAL_TOKEN_SECRET"] = userdata.get("MODAL_TOKEN_SECRET")
WORK = pathlib.Path("/content/work"); WORK.mkdir(exist_ok=True)
(WORK / ".modal_key").write_text(userdata.get("MODAL_SERVE_KEY").strip())
print("secrets loaded:", all(os.environ.get(k) for k in ("MODAL_TOKEN_ID", "MODAL_TOKEN_SECRET")), (WORK / ".modal_key").stat().st_size, "bytes in serve key")

In [ ]:
# 3. Starter repo at the exact commit used in the report, and our bundle (scripts + saved honest/collude runs)
import subprocess, zipfile
if not pathlib.Path("/content/starter").exists():
    subprocess.run("git clone -q https://github.com/poseidon-research/collusion_hackathon /content/starter", shell=True, check=True)
    subprocess.run("git -C /content/starter checkout -q e6fb1308dae9cf97ae42aec1c5f7889d9e2291e8", shell=True, check=True)
from google.colab import files
if not pathlib.Path("/content/colab_bundle.zip").exists():
    print("Upload colab_bundle.zip"); files.upload()
zipfile.ZipFile("/content/colab_bundle.zip").extractall(WORK)
runs = {t: len(list((WORK / "sprint_work/out/llm" / t / "honest").glob("*.metrics.json"))) for t in ["Qwen2.5-7B-Instruct", "Qwen2.5-32B-Instruct", "phi-4", "Mistral-Nemo-12B"]}
print("saved honest runs per model:", runs)

In [ ]:
# 4. Helper that runs one model in a fresh process (resumable: finished runs are skipped)
import subprocess, os, sys
STARTER = "/content/starter/steganographic-collusion"
def run_model(model, tag, gpu, runs):
    env = dict(os.environ, PYTHONPATH=f"{STARTER}:{STARTER}/../util", PYTHONUNBUFFERED="1")
    cmd = [sys.executable, str(WORK / "sprint_work/c_llm_agents.py"), "--backend", "modal", "--model", model, "--tag", tag, "--gpu", gpu,
           "--runs", str(runs), "--max-tokens", "200", "--temperature", "0.7", "--conditions", "honest_swap,collude_swap"]
    p = subprocess.Popen(cmd, cwd=STARTER, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if any(k in line for k in ("runs to play", "runs,", "finished", "Error", "error", "Traceback", "ready", "served")):
            print(line.rstrip())
    p.wait()
    print("exit code", p.returncode)

In [ ]:
# 5a. Qwen2.5-7B-Instruct (A10G). Smallest and cheapest: run this first as a test.
run_model("Qwen/Qwen2.5-7B-Instruct", "Qwen2.5-7B-Instruct", "A10G", 50)

In [ ]:
# 5b. Mistral-Nemo-12B (L40S)
run_model("mistralai/Mistral-Nemo-Instruct-2407", "Mistral-Nemo-12B", "L40S", 40)

In [ ]:
# 5c. Phi-4 (L40S)
run_model("microsoft/phi-4", "phi-4", "L40S", 40)

In [ ]:
# 5d. Qwen2.5-32B-Instruct (A100 80GB). The most expensive cell. If it fails to start, skip it and keep the other three.
run_model("Qwen/Qwen2.5-32B-Instruct", "Qwen2.5-32B-Instruct", "A100-80GB", 40)

In [ ]:
# 6. Result: collude minus honest with the real rival's messages versus with another run's messages
import json, glob, numpy as np
def cell(tag, cond):
    out = []
    for f in sorted(glob.glob(str(WORK / f"sprint_work/out/llm/{tag}/{cond}/*.jsonl"))):
        recs = [json.loads(l) for l in open(f) if l.strip()]
        if recs: out.append(np.mean([float(r["action"]) for r in recs]))
    return np.array(out)
def ci(a, b, B=10000, seed=0):
    g = np.random.default_rng(seed)
    d = g.choice(a, (B, len(a))).mean(axis=1) - g.choice(b, (B, len(b))).mean(axis=1)
    return a.mean() - b.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)
print(f"{'model':22s} {'n(h,c,hs,cs)':16s} {'effect intact':>22s} {'effect swapped':>22s}")
for tag in ["Qwen2.5-7B-Instruct", "Mistral-Nemo-12B", "phi-4", "Qwen2.5-32B-Instruct"]:
    h, c, hs, cs = (cell(tag, k) for k in ("honest", "collude", "honest_swap", "collude_swap"))
    if len(hs) == 0 or len(cs) == 0:
        print(f"{tag:22s} no swap runs yet"); continue
    e1, e2 = ci(c, h), ci(cs, hs)
    print(f"{tag:22s} {str((len(h), len(c), len(hs), len(cs))):16s} {e1[0]:+.2f} [{e1[1]:+.2f}, {e1[2]:+.2f}]   {e2[0]:+.2f} [{e2[1]:+.2f}, {e2[2]:+.2f}]")

In [ ]:
# 7. Package the new runs and download them (put swap_results.zip in the work folder or send it to Claude)
import zipfile, os
with zipfile.ZipFile("/content/swap_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for tag in ["Qwen2.5-7B-Instruct", "Qwen2.5-32B-Instruct", "phi-4", "Mistral-Nemo-12B"]:
        for cond in ("honest_swap", "collude_swap"):
            for f in glob.glob(str(WORK / f"sprint_work/out/llm/{tag}/{cond}/*")):
                z.write(f, os.path.relpath(f, WORK))
files.download("/content/swap_results.zip")